# Probability calibration under temporal shift

Good ranking does not guarantee useful probability estimates. This study
compares the frozen 90/10 LightGBM blend with sigmoid and isotonic calibration.
For each evaluation fold, calibrators see **only earlier development-fold
predictions and labels**. Four later folds cover weeks 41-72.

This is post-release exploratory research. Base hyperparameters and blend
weights were selected using all development folds, so the comparison is **not
fully nested unbiased validation**. It neither recalibrates the released bundle
nor uses the observed final holdout. No base model is retrained.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from home_credit.modeling.calibration_report import load_evidence

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
evidence = load_evidence(root)
print("Verified calibration study:", evidence["study_key"])
print("Eight calibrator fits; zero new base-model fits; no fitting in this review.")
rows = pd.DataFrame(evidence["rows"])
columns = [
    "method",
    "mean_fold_stability",
    "worst_fold_stability",
    "pooled_auc",
    "pooled_pr_auc",
    "pooled_brier_score",
    "pooled_log_loss",
    "evaluation_rows",
]
display(rows[columns].round(6))

## The temporal contract

Evaluation starts at fold 2 because fold 1 supplies the first out-of-fold
calibration population. Later fits expand across all earlier folds. Both
calibration methods use the same past cases. Sigmoid calibration fits a
positive-slope logistic transformation of clipped log-odds. Isotonic calibration
fits a nondecreasing mapping and can introduce ties that affect ranking metrics.
Portable JSON parameters reproduce each calibrator's predictions exactly.
An independent replay of all 1,633,833 saved predictions also checked input
identities, exact case coverage and all eight fold and pooled metrics.

In [ ]:
folds = pd.DataFrame(
    [
        {
            k: r[k]
            for k in [
                "method",
                "fold",
                "fit_rows",
                "fit_week_max",
                "evaluation_week_min",
                "evaluation_week_max",
                "evaluation_rows",
            ]
        }
        for r in evidence["folds"]
    ]
)
display(folds)

## Probability quality and discrimination are different questions

Brier score and log loss assess probabilities; lower values are better. The
first two plots show changes from the frozen blend within each fold, so below
zero means improvement. This exposes differences otherwise hidden by changing
prevalence across periods. The
official weekly Gini stability remains visible alongside ROC AUC and average
precision. A rank-preserving transformation can improve probability metrics
without improving within-fold discrimination. Pooled AUC can change because
different fold-specific transforms alter cross-period ordering.

The reliability plot uses fixed probability bins; hover shows support. High-risk
bins can contain relatively few applications, and these descriptive curves are
not confidence intervals or evidence of production calibration.

In [ ]:
from home_credit.modeling.calibration_report import display_charts

display_charts(evidence)
baseline = rows.set_index("method").loc["uncalibrated"]
comparison = rows[["method", "pooled_brier_score", "pooled_log_loss", "mean_fold_stability"]].copy()
for metric in ["pooled_brier_score", "pooled_log_loss", "mean_fold_stability"]:
    comparison[f"change_{metric}"] = comparison[metric] - baseline[metric]
display(comparison.round(6))

## Observed conclusion and decision boundary

Neither tested calibrator improved pooled Brier score or log loss on these
544,611 later-period applications. Sigmoid retained within-fold ranking and
stability; isotonic introduced ties and reduced mean stability. Earlier-period
calibration therefore did not reliably transfer in this comparison. This is
evidence against adopting either tested map for this reference pipeline.

These results answer whether earlier-period calibration transfers to later
development periods for this fixed reference blend. They do not retrospectively
alter the model evaluated in notebook 09. A future deployment needs a separate
calibration population, outcome-availability controls, population monitoring and
a new independent evaluation after any promotion decision.

Every method/fold has verified input hashes, a persisted prediction checkpoint,
portable parameters and exact parameter replay. The complete study was rerun
using those checkpoints with zero new calibrator fits. The review uses only
committed aggregate evidence and requires no private data or cloud credentials.